# Importations

In [1]:
# Numerical and scientific python programming
import numpy as np

# Local importations
from moments.bloch import generate_pauli_basis, compute_tensor_basis, compute_subset_index_map
from moments.optimization import opt_moment_preserving_ent

# Optimization

In [2]:
dn, N = 2, 2
dim = [dn] * N
d = 2 ** N

pauli_basis = generate_pauli_basis()
local_bases = [pauli_basis.copy()] * N
local_basis_sizes = [len(basis) for basis in local_bases]

tensor_basis = compute_tensor_basis(local_bases)
subset_index_map = compute_subset_index_map(local_basis_sizes)

Rt = {(1,): 0.5, (2,): 0.5, (1, 2): 1}

In [3]:
optimization_result = opt_moment_preserving_ent(dim, tensor_basis, subset_index_map, Rt,
                                                optimization="minimize", metric="partial_trace_norm", cholesky_opt=True, exact_jac=True)

## Validation

In [4]:
checks = optimization_result.checks

print("Is the output a valid density matrix?", checks["is_valid_dm"])

bloch_equal = []
bloch_diff = {}
moments_equal = []
for subset in optimization_result.bloch_initial.keys():
        bloch_equal.append(np.allclose(optimization_result.bloch_initial[subset], optimization_result.bloch_final[subset]))
        bloch_diff[subset] = float(np.linalg.norm(optimization_result.bloch_initial[subset] - optimization_result.bloch_final[subset]))
        moments_equal.append(np.allclose(optimization_result.moments_initial[subset], optimization_result.moments_final[subset]))

print("\nAre the density matrices equal?", np.allclose(optimization_result.rho_initial, optimization_result.rho_final))
print("Density matrix difference:", np.linalg.norm(optimization_result.rho_initial - optimization_result.rho_final))

print("\nAre the Bloch vectors equal?", all(bloch_equal))
print("Bloch vector difference:", bloch_diff)

print("\nAre the Bloch lengths equal?", checks["moments_equal"])
print("Bloch lengths difference?", checks["moments_distance"])


print("\nIs the metric equal?", np.isclose(optimization_result.metric_initial, optimization_result.metric_final))
print("Metric difference:", abs(optimization_result.metric_initial - optimization_result.metric_final))

Is the output a valid density matrix? True

Are the density matrices equal? False
Density matrix difference: 0.5499688076147671

Are the Bloch vectors equal? False
Bloch vector difference: {(1,): 0.48271348635122163, (2,): 0.4188696479550195, (1, 2): 0.8952087273443672}

Are the Bloch lengths equal? True
Bloch lengths difference? {(1,): 5.60218538225854e-13, (2,): 1.454392162258955e-13, (1, 2): 2.5279778270714814e-13}

Is the metric equal? False
Metric difference: 0.34938900483222723


In [5]:
optimizer_info = optimization_result.optimizer_info
print("Result success:", optimizer_info["result_success"])
print("Result message:", optimizer_info["result_message"])

Result success: True
Result message: Optimization terminated successfully


In [6]:
print(optimization_result.metric_initial)
print(optimization_result.metric_final)

1.349389004832738
1.0000000000005107
